## Native SDK example 

In [27]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

client = genai.Client(
    api_key=os.getenv("GOOGLE_API_KEY")
)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Explain Retrieval Augmented Generation in 3 sentences."
)

print(response.text)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Retrieval Augmented Generation (RAG) enhances Large Language Models (LLMs) by giving them access to external, factual knowledge beyond their original training data. Before generating a response, RAG first retrieves relevant information or documents from a knowledge base based on the user's query. This retrieved context is then provided to the LLM, enabling it to produce more accurate, up-to-date, and less "hallucinatory" answers grounded in specific, verifiable data.


## Streaming Output

In [29]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

client = genai.Client(
    api_key=os.getenv("GOOGLE_API_KEY")
)

response = client.models.generate_content_stream(
    model="gemini-2.5-flash",
    contents="Explain how a transformer works in simple terms."
)

for chunk in response:
    if chunk.text:
        print(chunk.text, end="", flush=True)

print()

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Imagine a transformer as an electrical "gearbox" for voltage. It doesn't have moving parts, but it uses magnetism to change the voltage of electricity.

Here's how it works in simple terms:

1.  **Two Coils, No Contact:** A transformer has two separate coils of wire, called the **primary coil** and the **secondary coil**. These coils are NOT electrically connected to each other. They are usually wound around a common piece of iron (called the core).

2.  **Input (Primary Coil):** When you plug the transformer into an AC (Alternating Current) power source (like your wall outlet), the electricity flows into the **primary coil**.

3.  **Creating a Changing Magnetic Field:** Because the current is AC, it constantly changes direction and strength. This *changing* current flowing through the primary coil creates a *changing* magnetic field around it. Think of it like a magnet that's constantly getting stronger, weaker, and flipping its poles.

4.  **The Core's Job:** The iron core is really 

## Model Configurations and Controls

In [31]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()

client = genai.Client(
    api_key=os.getenv("GOOGLE_API_KEY")
)

config = types.GenerateContentConfig(
    # Controls randomness.
    # Lower = more deterministic, higher = more creative/varied.
    temperature=0.3,

    # Limits sampling to the smallest set of tokens
    # whose cumulative probability reaches this value.
    # Higher = more possible token choices.
    top_p=0.9,

    # Limits sampling to the top K most probable tokens.
    # Lower = more focused, higher = more diverse.
    top_k=40,

    # Maximum number of tokens the model can generate.
    max_output_tokens=500,

    # Defines the model's role and general behavior.
    system_instruction=(
        "You are an AI engineering instructor. "
        "Give concise, technically accurate explanations "
        "with practical examples."
    ),
)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="What is the difference between RAG and fine-tuning?",
    config=config,
)

print(response.text)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Both RAG (Retrieval-Augmented Generation) and fine-tuning aim to


`temperature`, `top_p`, and `top_k` all influence **how the model selects the next token**, but they work differently. **Temperature** controls how much the probability distribution is flattened or sharpened—lower values favor predictable tokens, while higher values allow more variation. **Top-k** limits selection to the **K most probable tokens**, while **top-p** dynamically selects the smallest group of tokens whose cumulative probability reaches `p`. In simple terms, temperature controls **how random the selection is**, while top-k and top-p control **which tokens are eligible for selection**. `max_output_tokens` is different: it limits **how much the model can generate**, rather than how it chooses tokens. `system_instruction` also operates at a different level, defining the model's **overall behavior and role** rather than directly controlling token sampling.

A useful teaching diagram is:

```text
                 Model generates next token
                          │
                 ┌────────▼────────┐
                 │ Probability     │
                 │ distribution    │
                 └────────┬────────┘
                          │
              ┌───────────┴───────────┐
              │                       │
          Top-K / Top-P          Temperature
          Which tokens?          How random?
              │                       │
              └───────────┬───────────┘
                          │
                    Select token
                          │
                          ▼
                    Next token ...
                          │
                    Repeat until
                    output limit
                          │
                   max_output_tokens
```

**Rule of thumb:** start with `temperature`, `top_p`, and `top_k` at reasonable defaults; tune them only when you have a specific output-quality problem to solve.


### How Temperature Works? 
Temperature is applied to the model's **next-token probability distribution** before the model samples a token.

Suppose the model predicts the next word with these probabilities:

```text
"Paris"     70%
"London"    20%
"Berlin"     7%
"Tokyo"      3%
```

Temperature changes these probabilities using the **softmax temperature equation**:

$$
P_i = \frac{e^{z_i/T}}{\sum_j e^{z_j/T}}
$$

where:

* \(z_i\) = model's original **logit** for token \(i\)
* \(T\) = temperature
* \(P_i\) = resulting probability

### Low temperature

When:

```text
T < 1
```

the probability distribution becomes **sharper**:

```text
Paris     95%
London     4%
Berlin     1%
Tokyo      0%
```

So the model strongly favors the most likely token.

```text
Temperature ↓
      ↓
Distribution becomes sharper
      ↓
High-probability tokens dominate
      ↓
More predictable output
```

### Temperature = 1

At:

```text
T = 1
```

the logits are effectively unchanged before softmax:

```text
Paris     70%
London    20%
Berlin     7%
Tokyo      3%
```

### High temperature

When:

```text
T > 1
```

the distribution becomes **flatter**:

```text
Paris     45%
London    28%
Berlin    17%
Tokyo     10%
```

Now less-probable tokens have a greater chance of being selected.

```text
Temperature ↑
      ↓
Distribution becomes flatter
      ↓
More tokens have meaningful probability
      ↓
More varied / less predictable output
```

### The key misconception

Temperature **does not directly tell the model to be "more creative."**

It modifies the **probability distribution used during token sampling**.

For example:

```text
                 Original logits
                       │
                       ▼
                  Temperature
                       │
                       ▼
              Probability distribution
                       │
                       ▼
                 Token sampling
                       │
                       ▼
                  Next token
```

So when you set:

```python
temperature=0.2
```

you're essentially saying:

> "Make the probability distribution more concentrated around the model's preferred tokens."

Whereas:

```python
temperature=1.5
```

means:

> "Flatten the distribution so lower-probability tokens have a better chance of being selected."

### Why this matters

Consider a model choosing the next word:

```text
"The capital of France is ___"

Paris       99%
London       0.5%
Berlin       0.3%
Rome         0.2%
```

Increasing temperature doesn't magically make the model *know more*. It makes the sampling process more willing to deviate from its highest-probability choice.

For factual tasks:

```text
Lower temperature → usually more consistent
```

For creative generation:

```text
Higher temperature → potentially more varied
```

One important engineering detail: **temperature interacts with `top_k` and `top_p`**. Those parameters can first restrict which tokens are eligible, while temperature changes the probabilities among the remaining candidates.


## Unified Model Gateway
                Your Application
                           │
                           ▼
                 ┌───────────────────┐
                 │   Model Gateway    │
                 │                   │
                 │ OpenRouter /      │
                 │ LiteLLM           │
                 └─────────┬─────────┘
                           │
             ┌─────────────┼─────────────┐
             ▼             ▼             ▼
          OpenAI        Gemini       Anthropic
             │             │             │
          GPT-5.x       Gemini       Claude

In [30]:
from openai import OpenAI

client = OpenAI(
    api_key="anything",
    base_url="http://localhost:4000"
)

response = client.chat.completions.create(
    model="gemini-primary",
    messages=[
        {
            "role": "user",
            "content": "Explain Retrieval Augmented Generation."
        }
    ]
)

print(response.choices[0].message.content)

**Retrieval Augmented Generation (RAG)** is an AI framework that enhances the capabilities of large language models (LLMs) by providing them with access to external, up-to-date, and domain-specific information during the generation process.

Think of it like this: an LLM, on its own, is like a brilliant student who has read many books in their life but can only rely on what they *remember*. RAG is like giving that student an "open book exam" – they can quickly look up specific information from a vast library before answering a question.

### Why is RAG Needed? (Problems it Solves)

Large Language Models (LLMs) like GPT-3/4, Bard, Llama, etc., are trained on enormous datasets, but they have several inherent limitations:

1.  **Knowledge Cutoff:** Their knowledge is limited to their training data. They don't know about events or facts that occurred after their last training update.
2.  **Hallucination:** LLMs can sometimes generate plausible-sounding but factually incorrect or nonsensica

## Strcutured Output

In [2]:
paragraph = """
John Smith is a Senior Machine Learning Engineer at Acme AI. He has
6 years of experience in machine learning and specializes in natural
language processing, large language models, and retrieval augmented
generation. He holds a Master's degree in Computer Science from
Stanford University and is based in San Francisco. John is currently
working on an AI-powered document processing platform.
"""

### Via Prompt

In [32]:
from openai import OpenAI
import json

client = OpenAI(
    api_key="anything",
    base_url="http://localhost:4000"
)

response = client.chat.completions.create(
    model="gemini-primary",
    messages=[
        {
            "role": "system",
            "content": """
Extract candidate information from the provided text.

Return the result as JSON with these fields:
- name: string 
- job_title
- company
- experience_years
- skills
- education
- university
- location
- current_project
"""
        },
        {
            "role": "user",
            "content": paragraph
        }
    ],
    response_format={"type": "json_object"}
)

result = json.loads(response.choices[0].message.content)

print(json.dumps(result, indent=2))

{
  "name": "John Smith",
  "job_title": "Senior Machine Learning Engineer",
  "company": "Acme AI",
  "experience_years": "6",
  "skills": [
    "natural language processing",
    "large language models",
    "retrieval augmented generation"
  ],
  "education": "Master's degree in Computer Science",
  "university": "Stanford University",
  "location": "San Francisco",
  "current_project": "AI-powered document processing platform"
}


In [36]:
from openai import OpenAI
import json

client = OpenAI(
    api_key="anything",
    base_url="http://localhost:4000"
)

paragraph = """
John Smith is a Senior Machine Learning Engineer at Acme AI. He has
6 years of experience in machine learning and specializes in natural
language processing, large language models, and retrieval augmented
generation. He holds a Master's degree in Computer Science from
Stanford University and is based in San Francisco. John is currently
working on an AI-powered document processing platform.
"""

response_schema = {
    "type": "object",
    "properties": {
        "name": {
            "type": "string"
        },
        "job_title": {
            "type": "string"
        },
        "company": {
            "type": "string"
        },
        "experience_years": {
            "type": "integer"
        },
        "skills": {
            "type": "array",
            "items": {
                "type": "string"
            }
        },
        "education": {
            "type": "string"
        },
        "university": {
            "type": "string"
        },
        "location": {
            "type": "string"
        },
        "current_project": {
            "type": "string"
        },
        "best_project": {
                "type": "string"
        }
        
    },
    "required": [
        "name",
        "job_title",
        "company",
        "experience_years",
        "skills",
        "education",
        "university",
        "location",
        "current_project",
        "best_project"
    ],
    "additionalProperties": False
}

response = client.chat.completions.create(
    model="gemini-primary",
    messages=[
        {
            "role": "system",
            "content": "Extract candidate information from the provided text."
        },
        {
            "role": "user",
            "content": paragraph
        }
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "candidate",
            "strict": True,
            "schema": response_schema
        }
    }
)

result = json.loads(
    response.choices[0].message.content
)

print(json.dumps(result, indent=2))

{
  "name": "John Smith",
  "job_title": "Senior Machine Learning Engineer",
  "company": "Acme AI",
  "experience_years": 6,
  "skills": [
    "natural language processing",
    "large language models",
    "retrieval augmented generation"
  ],
  "education": "Master's degree",
  "university": "Stanford University",
  "location": "San Francisco",
  "current_project": "AI-powered document processing platform",
  "best_project": "Not specified"
}


### Pydantic Validation

In [39]:
import os
from dotenv import load_dotenv

from google import genai
from google.genai import types

from pydantic import BaseModel


load_dotenv()

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [40]:
class Candidate(BaseModel):
    name: str
    job_title: str
    company: str
    experience_years: int
    skills: list[str]
    education: str
    university: str
    location: str
    current_project: str

In [42]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=f"""
Extract the candidate information from the following paragraph.

Paragraph:
{paragraph}
""",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=Candidate,
    ),
)

In [43]:
print(response.text)

{
  "name": "John Smith",
  "job_title": "Senior Machine Learning Engineer",
  "company": "Acme AI",
  "experience_years": 6,
  "skills": ["natural language processing", "large language models", "retrieval augmented generation"],
  "education": "Master's degree in Computer Science",
  "university": "Stanford University",
  "location": "San Francisco",
  "current_project": "an AI-powered document processing platform"
}


## Prompt Engineering VS Context Engineering

**Prompt engineering** is the practice of writing clear instructions that guide an AI toward a desired output.

**Context engineering** is the practice of providing the AI with the right background information, data, examples, tools, and conversation history so it can respond accurately.

### Prompt engineering tips

- State the task clearly: “Summarize this article in  five bullet points.”
- Specify the desired format: table, JSON, checklist, paragraph, or code.
- Define the audience and tone.
- Include constraints such as length, language, or required sections.
- Provide examples of good input and output.
- Break complex tasks into smaller steps.
- Ask the model to identify missing information instead of guessing.
- Use precise verbs: *compare, classify, extract, rewrite, evaluate*.
- Separate instructions from source material with delimiters such as `<text>...</text>`.
- Refine prompts based on the model’s previous output.

### Context engineering tips

- Provide only relevant information; excessive context can reduce focus.
- Put the most important instructions and facts where the model can easily identify them.
- Include definitions for domain-specific terms.
- Keep facts, instructions, examples, and user data clearly separated.
- Use retrieved documents or databases for information that changes frequently.
- Remove outdated, duplicated, or contradictory context.
- Include metadata such as source, date, author, and reliability when relevant.
- Limit conversation history to what is necessary.
- Use structured formats such as JSON or XML for complex data.
- Tell the model how to handle conflicting sources or missing information.
- Summarize long conversations before adding them to the context.

A useful distinction is: **prompt engineering shapes the instruction; context engineering shapes the information surrounding the instruction.**

In [44]:
from google import genai

client = genai.Client()

def summarize(text: str, audience: str = "a general reader") -> str:
    prompt = f"""
Task: Summarize the source text:.

Audience:
{audience}

Output requirements:
- Use 3 to 5 bullet points.
- Use clear, concise language.
- Include only information supported by the source.
- Mention uncertainty when the source is unclear.
- End with a one-sentence conclusion.

Source text:
<source>
{text}
</source>

"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )

    return response.text


article = """
Cloud computing provides on-demand access to computing resources.
It can reduce infrastructure costs and improve scalability, but it
also creates security, compliance, and vendor-dependency risks.
"""

print(summarize(article, "a small-business owner"))


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Here's a summary for a small-business owner:

*   Cloud computing gives you on-demand access to computing resources.
*   It can potentially lower your infrastructure costs and improve your business's ability to scale.
*   However, it also brings new risks related to security, regulatory compliance, and relying too heavily on a single vendor.

Consider these potential benefits and risks when exploring cloud solutions for your business.


In [17]:
from typing import List, Literal
from pydantic import BaseModel, Field
from google import genai
from google.genai import types
import json


# -------------------------
# 1. Define the response schema
# -------------------------

class TroubleshootingStep(BaseModel):
    step: int = Field(description="The step number, starting at 1")
    instruction: str = Field(description="A clear action for the customer")
    purpose: str = Field(description="Why this action may help")


class LikelyExplanation(BaseModel):
    explanation: str
    confidence: Literal["low", "medium", "high"]
    reasoning: str


class SupportAnalysis(BaseModel):
    case_summary: str
    confirmed_facts: List[str]
    likely_explanation: LikelyExplanation
    recommended_troubleshooting: List[TroubleshootingStep]
    customer_reply: str = Field(
        description="A professional customer-facing reply under 180 words"
    )
    internal_next_action: str
    missing_information: List[str]


# -------------------------
# 2. Create the client
# -------------------------

client = genai.Client()


# -------------------------
# 3. Prepare the context
# -------------------------

context = {
    "company": {
        "name": "Northstar Cloud",
        "product": "Northstar Drive",
        "support_tone": "professional, friendly, and concise"
    },
    "customer": {
        "name": "Maya",
        "plan": "Professional",
        "account_age": "2 years",
        "customer_sentiment": "frustrated but cooperative"
    },
    "support_policy": {
        "refund_window_days": 30,
        "professional_plan_sla_hours": 4,
        "maximum_credit_without_manager_approval": 50,
        "allowed_actions": [
            "Explain the issue clearly",
            "Provide troubleshooting steps",
            "Escalate technical problems",
            "Offer a service credit up to $50",
            "Ask for logs or screenshots"
        ],
        "not_allowed": [
            "Promise a specific fix date",
            "Blame the customer",
            "Invent account details",
            "Offer more than $50 without approval",
            "Claim that an issue is fixed without evidence"
        ]
    },
    "product_information": {
        "file_sync_requirements": [
            "The desktop application must be running",
            "The user must be signed in",
            "The folder must be included in sync settings",
            "The device must have an internet connection"
        ],
        "known_issue": {
            "title": "Delayed synchronization for large folders",
            "status": "Investigating",
            "affected_versions": ["5.2.0", "5.2.1"],
            "workaround": (
                "Pause and resume synchronization, then restart the "
                "desktop application"
            )
        }
    },
    "conversation_history": [
        {
            "speaker": "customer",
            "message": (
                "My project folder has not synced for six hours. "
                "I restarted my computer, but the files still do not "
                "appear on my laptop."
            )
        },
        {
            "speaker": "agent",
            "message": (
                "Can you confirm whether the Northstar Drive icon is "
                "visible in your system tray?"
            )
        },
        {
            "speaker": "customer",
            "message": (
                "Yes. The folder shows a spinning sync icon. "
                "The folder contains about 18,000 files."
            )
        }
    ],
    "technical_notes": [
        "The customer is using desktop application version 5.2.1.",
        "The folder contains approximately 18,000 files.",
        "No error message has been reported.",
        "The customer has already restarted the computer."
    ]
}


# -------------------------
# 4. Write the prompt
# -------------------------

prompt = f"""
You are a senior customer-support assistant for Northstar Cloud.

Analyze the support case contained in the context and produce:
- A concise case summary
- The confirmed facts
- The most likely explanation
- Recommended troubleshooting steps
- A customer-facing response
- The next internal support action
- Any missing information

Rules:
- Use only information from the context.
- Do not invent technical details, account information, or timelines.
- Treat the known issue as a possible explanation, not a confirmed cause.
- Do not recommend restarting the computer as the first step because the
  customer has already done that.
- Do not promise a specific fix date.
- Do not blame the customer.
- Do not offer a service credit automatically.
- Do not ask for information the customer already provided.
- Keep the customer-facing reply below 180 words.
- Clearly distinguish confirmed facts from assumptions.
- If information is insufficient, list it under missing_information.

The response must follow the supplied response schema.

<context>
{json.dumps(context, indent=2)}
</context>
"""


# -------------------------
# 5. Generate schema-constrained output
# -------------------------

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
    config=types.GenerateContentConfig(
        temperature=0.2,
        response_mime_type="application/json",
        response_schema=SupportAnalysis,
    ),
)


# -------------------------
# 6. Parse the typed response
# -------------------------

analysis: SupportAnalysis = response.parsed

print("CASE SUMMARY:")
print(analysis.case_summary)

print("\nCONFIRMED FACTS:")
for fact in analysis.confirmed_facts:
    print(f"- {fact}")

print("\nLIKELY EXPLANATION:")
print(analysis.likely_explanation.explanation)
print("Confidence:", analysis.likely_explanation.confidence)

print("\nTROUBLESHOOTING:")
for item in analysis.recommended_troubleshooting:
    print(f"{item.step}. {item.instruction}")
    print(f"   Purpose: {item.purpose}")

print("\nCUSTOMER REPLY:")
print(analysis.customer_reply)

print("\nINTERNAL NEXT ACTION:")
print(analysis.internal_next_action)

print("\nMISSING INFORMATION:")
for item in analysis.missing_information:
    print(f"- {item}")

CASE SUMMARY:
Customer Maya is experiencing a prolonged synchronization delay for a project folder containing approximately 18,000 files. The Northstar Drive desktop application (version 5.2.1) shows a spinning sync icon, and a computer restart did not resolve the issue.

CONFIRMED FACTS:
- The customer's name is Maya and she is on a Professional plan.
- Her account is 2 years old and her sentiment is frustrated but cooperative.
- Her project folder has not synced for six hours.
- She has already restarted her computer.
- The Northstar Drive icon is visible in her system tray.
- The folder shows a spinning sync icon.
- The folder contains approximately 18,000 files.
- The customer is using desktop application version 5.2.1.
- No error message has been reported.

LIKELY EXPLANATION:
The delayed synchronization of the large project folder is likely due to a known issue affecting Northstar Drive desktop application versions 5.2.0 and 5.2.1, which causes delays when syncing a high volume o